# Strands: MCP y A2A

Input → Agent con runtime explícito → Tool y RunResult. Después, el laboratorio muestra la capacidad distintiva del framework. Instala el extra del framework y Studio para la segunda sección. No hay fallback de proveedor.

In [ ]:
import agentic_systems as toolkit
assert toolkit.__version__ == "2.1.3"

In [ ]:
@toolkit.tool
def certify(value: int) -> dict:
    """Conserva el entero como evidencia exacta."""
    return {"value": value, "verified": True}

evidence = toolkit.skill(name="evidence", tools=(certify,))

In [ ]:
runtime = toolkit.runtime(provider="python-runtime")
framework = 'strands'

In [ ]:
actor = toolkit.Agent(
    "verification", runtime=runtime, framework=toolkit.framework(framework),
    skills=(evidence,), instructions="Call certify exactly once with value=37.",
    contract={"tool_expectation": {"all_of": ["certify"]},
              "completion": "when_required_tools_satisfied"},
    policy={"repair": False},
)
request = {"tool": "certify", "input": {"value": 37}} if runtime.provider == "python-runtime" else "Call certify with value=37."
result = actor.run(request, limits=toolkit.Limits(timeout_s=180))
assert result.ok and result.check_invariants().ok, result.to_dict()
events = [event for node in result.walk() for event in node.tool_events]
assert any(e.name == "certify" and e.ok and e.input.get("value") == 37 for e in events)
print({"input": request, "process": "runtime + framework → certify → Tool evidence", "output": result.data})
print(result.lineage(name="verification").explain())

La siguiente sección reutiliza el laboratorio instalado de **Studio 2.1.3**, distribuido por separado. Aquí se prueba la capacidad específica del SDK; la API común anterior sigue siendo Agent(runtime=..., framework=...). [Implementación completa](../../examples/agentic_systems_studio/src/agentic_systems_studio/lab/probes.py). En modo Python se verifica el protocolo con un modelo determinista; el modelo externo se valida en la campaña live.

In [ ]:
from agentic_systems_studio.lab.probes import protocols
for protocol in ("mcp", "a2a"):
    report = protocols("python-runtime", protocol, None)
    assert report["result"]["ok"]
    print(report["protocol"])